# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1: "High scroll + high engagement = +11.2 health points" (Finding #5)**
* **Methodology Question:** How is Health Score defined? The paper's methodology states that Health Score is calculated as "Impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts)". If scroll depth makes up 20% of the Health Score math, comparing scroll behavior to Health Score is a circular claim. This violates the "Label-derived features" rule from our leakage taxonomy, as the feature is mathematically baked into the outcome.

**Finding 2: "Growing content is 37.6% longer and 20% younger" (Finding #1)**
* **Methodology Question:** How do the time windows align? "Growing" is defined as a >10% increase in impressions in the last 30 days vs the previous 30 days. Was the word count measured *before* the 60-day window began, or is it a snapshot from today? If a page was updated and lengthened *during* the growth window, using today's word count to explain past growth is a future-window leakage violation.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*To audit my own K-Means model from Week 5, I am comparing a standard Random Split against an honest Grouped Split (isolated by `client_hash_id`). Random splits allow the model to memorize the specific impression/CTR behaviors of massive clients, artificially inflating how cleanly the clusters overlap with the baseline. The grouped split forces the model to prove its clusters generalize to entirely unseen clients.*

In [2]:
import duckdb
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from google.colab import userdata

# 1. Load Data
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet"

query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,
    SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr,
    AVG(gsc_avg_position) AS avg_position
FROM read_parquet('{fact_path}')
GROUP BY 1, 2
HAVING SUM(gsc_impressions) > 100
LIMIT 150000
"""
df = con.sql(query).df()

# Baseline Rule (Target for overlap)
visible = (df['total_impressions'] >= 1000).astype(int)
good_rank = (df['avg_position'] <= 20).astype(int)
low_ctr = (df['ctr'] < 0.02).astype(int)
df['is_bleeding_baseline'] = visible * good_rank * low_ctr

features = ['total_impressions', 'ctr', 'avg_position']

def run_clustering_audit(train_df, test_df, split_name):
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('kmeans', KMeans(n_clusters=4, random_state=42, n_init=10))
    ])

    pipeline.fit(train_df[features])
    test_df = test_df.copy()
    test_df['cluster'] = pipeline.predict(test_df[features])

    # Identify the anomaly cluster (highest impressions, lowest CTR)
    stats = test_df.groupby('cluster').agg(
        med_imp=('total_impressions', 'median'),
        med_ctr=('ctr', 'median')
    )
    target_c = stats.sort_values(by=['med_imp', 'med_ctr'], ascending=[False, True]).index[0]

    # Calculate overlap with baseline
    test_df['in_target_cluster'] = (test_df['cluster'] == target_c).astype(int)
    overlap = test_df[test_df['is_bleeding_baseline'] == 1]['in_target_cluster'].mean()
    print(f"{split_name} Overlap with Baseline: {overlap:.1%}")

# --- BEFORE: Dishonest Random Split ---
train_rand, test_rand = train_test_split(df, test_size=0.25, random_state=42)
run_clustering_audit(train_rand, test_rand, "[BEFORE] Random Split")

# --- AFTER: Honest Grouped Split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))
train_grp, test_grp = df.iloc[train_idx], df.iloc[test_idx]
run_clustering_audit(train_grp, test_grp, "[AFTER] Grouped Split (by Client)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[BEFORE] Random Split Overlap with Baseline: 4.2%
[AFTER] Grouped Split (by Client) Overlap with Baseline: 0.5%


## 3. Leakage audit

**The Attack-Your-Own-Model Audit:**
* **Label-derived features:** FAILED. In Week 5, I evaluated my K-Means model by comparing its clusters against a heuristic baseline score. However, that baseline was calculated using `total_impressions`, `ctr`, and `avg_position`—the exact same columns fed into the K-Means model. This is circular validation; the model and the baseline share the exact same mathematical inputs.
* **Future/overlapping windows:** FAILED. The features were aggregated over the entire month of March 2026, and the baseline evaluation occurred over that exact same window. There is no separation between the feature-learning window and the outcome window.
* **Conclusion:** While unsupervised clustering doesn't "cheat" on a label during training, validating it against a rule built from the exact same inputs means the overlap score is an inevitable math artifact, not a discovery.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

**Original bold claim (from Week 5):**
*"The K-Means approach proved superior at mathematically separating extreme outliers from moderate-volume content, successfully catching the bleeding impressions."*

**Rewritten safe claim (Decision-Support Language):**
*"Within this specific March 2026 dataset, we observed that an unsupervised K-Means model grouped content into a distinct high-visibility, low-engagement cluster. While this overlap is partly due to shared input features, the clustering approach offers directional decision-support for prioritizing metadata reviews without relying on hardcoded thresholds."*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.